# 04 · Regularization: Ridge, Lasso & Elastic Net
> **Part:** Part 1 · Classical supervised learning  
> **Family:** `Supervised · Regression (penalized linear models)`

## Definition
Regularization adds a penalty on the size of the weights to the loss so the model cannot over-use any feature.
**Ridge (L2)** shrinks all weights smoothly toward zero; **Lasso (L1)** can set weights *exactly* to zero (automatic
feature selection); **Elastic Net** mixes both.

## Intuition
With many correlated or noisy features, unpenalized regression happily assigns large opposing weights that fit noise.
The penalty is a budget: weights must "pay" for being large, so only features that truly help earn a big weight. The L1 penalty
has corners on the axes, so the optimum often lands exactly on a corner, which means a zero weight.


## The Math

```text
minimize   1/(2n) ||y − Xw||²  +  λ [ α·||w||₁ + (1−α)/2·||w||₂² ]
    Ridge (α=0): w = (XᵀX/n + λI)⁻¹ Xᵀy/n
    Lasso/Elastic Net via coordinate descent:  wⱼ ← S(ρⱼ, λα) / (1 + λ(1−α)),
    ρⱼ = 1/n Xⱼᵀ(residual + Xⱼwⱼ),  S(r,t) = sign(r)·max(|r|−t, 0)   (soft-thresholding)
```


In [ ]:
# Setup, Imports & Styling
%matplotlib inline
import os
import pathlib
import sys
from IPython.display import Image, display

# Ensure parent folder is in path to import mlatlas
parent_dir = pathlib.Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlatlas import data, viz, cli

# Ensure interactive inline mode and configure Seaborn theme
plt.ion()
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 100, "figure.facecolor": "white"})


### Step 1: Feature Matrix & Correlation Exploration with Seaborn
We examine a house prices dataset extended with 6 pure-noise features to simulate real-world high-dimensional regression with noisy predictors.


In [ ]:
rng = np.random.default_rng(0)
X, y, names = data.house_prices(300, seed=1)
X = np.c_[X, rng.normal(size=(300, 6))]
names = names + [f"noise{i}" for i in range(1, 7)]

plt.figure(figsize=(9, 6.5))
corr_matrix = np.corrcoef(X.T)
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="vlag", xticklabels=names, yticklabels=names)
plt.title("Feature Correlation Matrix: 4 Informative Features + 6 Pure Noise Features", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


### Step 2: From-Scratch Algorithm Implementation
Below is the complete implementation of the model components in pure NumPy.


In [ ]:
# Helper Classes & Mathematical Logic
def soft(r, t): return np.sign(r) * np.maximum(np.abs(r) - t, 0)
def enet(X, y, lam, alpha, iters=150):
    n, d = X.shape; w = np.zeros(d); r = y.copy()
    for _ in range(iters):
        for j in range(d):
            r += X[:, j] * w[j]; w[j] = soft(X[:, j] @ r / n, lam * alpha) / (1 + lam * (1 - alpha)); r -= X[:, j] * w[j]
    return w


In [ ]:
# Main Algorithm & Training Routine
def main(gif, quick):
    rng = np.random.default_rng(0)
    X, y, names = data.house_prices(300, seed=1); X = np.c_[X, rng.normal(size=(300, 6))]
    names = names + [f"noise{i}" for i in range(1, 7)]
    Xtr, Xte, ytr, yte = data.train_test_split(X, y, test=.5, seed=2)
    Ztr, mu, sd = data.standardize(Xtr); Zte = (Xte - mu) / sd; ym, ys = ytr.mean(), ytr.std(); ytr_, yte_ = (ytr - ym) / ys, (yte - ym) / ys
    lams = np.logspace(-3, 1, 12 if quick else 40)
    ridge = np.array([enet(Ztr, ytr_, l, 0., 60) for l in lams]); lasso = np.array([enet(Ztr, ytr_, l, 1.) for l in lams[::-1]])[::-1]
    te = lambda W: [np.mean((Zte @ w - yte_) ** 2) for w in W]
    ter, tel = te(ridge), te(lasso)
    print(f"  best ridge λ={lams[np.argmin(ter)]:.3f} (test MSE {min(ter):.3f}),  best lasso λ={lams[np.argmin(tel)]:.3f} (test MSE {min(tel):.3f})")
    fig, (a1, a2, a3) = viz.subplots(1, 3, w=14, h=4.2)
    col = [viz.PALETTE[i] for i in range(4)] + ["#aaaaaa"] * 6

    def update(k):
        for a in (a1, a2, a3): a.clear()
        for ax, W, ttl in [(a1, ridge, "Ridge (L2): shrinks everything"), (a2, lasso, "Lasso (L1): zeroes out features")]:
            for j in range(10): ax.plot(lams, W[:, j], color=col[j], lw=2 if j < 4 else 1, label=names[j] if j < 4 else None)
            ax.axvline(lams[k], color="k", ls="--"); ax.set_xscale("log"); ax.set(title=ttl, xlabel="λ (regularization strength)", ylim=(-.2, 1.0))
        a1.legend(loc="upper right"); nz = int((np.abs(lasso[k]) > 1e-6).sum())
        a2.set_title(f"Lasso (L1): {nz} of 10 features alive at λ={lams[k]:.3f}")
        a3.semilogx(lams, ter, label="ridge", color=viz.PALETTE[0]); a3.semilogx(lams, tel, label="lasso", color=viz.PALETTE[1])
        a3.scatter([lams[k]] * 2, [ter[k], tel[k]], color="k", zorder=5); a3.set(title="Test error (U-shape = bias/variance)", xlabel="λ", ylabel="test MSE (standardized)"); a3.legend()
    viz.animate(fig, update, range(len(lams)), gif, fps=6)


### Step 3: Coefficient Regularization Paths (Ridge vs. Lasso)
We plot how feature weights shrink as the penalty parameter $\lambda$ increases. Notice that Lasso drives weights exactly to zero (sparse feature selection), whereas Ridge shrinks them smoothly.


In [ ]:
# Prepare standardized data splits
Xtr, Xte, ytr, yte = data.train_test_split(X, y, test=0.5, seed=2)
Ztr, mu, sd = data.standardize(Xtr)
Zte = (Xte - mu) / sd
ym, ys = ytr.mean(), ytr.std()
ytr_, yte_ = (ytr - ym) / ys, (yte - ym) / ys

lams = np.logspace(-3, 1, 30)
ridge = np.array([enet(Ztr, ytr_, l, 0.0, 60) for l in lams])
lasso = np.array([enet(Ztr, ytr_, l, 1.0) for l in lams[::-1]])[::-1]

te = lambda W: [np.mean((Zte @ w - yte_) ** 2) for w in W]
ter, tel = te(ridge), te(lasso)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.8))
colors = ["#4c78a8", "#f58518", "#54a24b", "#e45756"] + ["#aaaaaa"] * 6

# Panel 1: Ridge Paths
for j in range(10):
    ax1.plot(lams, ridge[:, j], color=colors[j], lw=2 if j < 4 else 1, label=names[j] if j < 4 else None)
ax1.set_xscale("log")
ax1.set_title("Ridge (L2): Smooth Shrinkage", fontsize=11, fontweight="bold")
ax1.set_xlabel("Regularization Strength (lambda)")
ax1.set_ylabel("Coefficient")
ax1.legend(loc="upper right", fontsize=8)

# Panel 2: Lasso Paths
for j in range(10):
    ax2.plot(lams, lasso[:, j], color=colors[j], lw=2 if j < 4 else 1, label=names[j] if j < 4 else None)
ax2.set_xscale("log")
ax2.set_title("Lasso (L1): Exact Feature Selection", fontsize=11, fontweight="bold")
ax2.set_xlabel("Regularization Strength (lambda)")

# Panel 3: Test Error U-curve
ax3.semilogx(lams, ter, label="Ridge MSE", color="#4c78a8", lw=2)
ax3.semilogx(lams, tel, label="Lasso MSE", color="#f58518", lw=2)
ax3.set_title("Validation Error (Bias-Variance U-Curve)", fontsize=11, fontweight="bold")
ax3.set_xlabel("Regularization Strength (lambda)")
ax3.set_ylabel("Test MSE (Standardized)")
ax3.legend()

plt.tight_layout()
plt.show()


### Step 4: Industry Benchmark (Comparison with scikit-learn)
We compare our from-scratch regularized models against `sklearn.linear_model.Ridge` and `sklearn.linear_model.Lasso`.


In [ ]:
from sklearn.linear_model import Ridge, Lasso

sk_ridge = Ridge(alpha=1.0).fit(Ztr, ytr_)
sk_lasso = Lasso(alpha=0.05).fit(Ztr, ytr_)
print("scikit-learn Ridge coefficients (first 4 informative):", sk_ridge.coef_[:4].round(3))
print("scikit-learn Lasso coefficients (first 4 informative):", sk_lasso.coef_[:4].round(3))
print("scikit-learn Lasso noise coefficients (should be ~0):", sk_lasso.coef_[4:].round(3))


### Step 5: Visualizing Learning Dynamics

We use the 4 real house-price features plus 6 pure-noise columns. As λ grows: left, ridge paths shrink gradually;
middle, lasso paths hit zero one by one, noise features first; right, test error forms a U shape.


In [ ]:
# Display the learning animation inline
media_file = f"../outputs/04_regularization_ridge_lasso.gif"
if os.path.exists(media_file):
    display(Image(filename=media_file))
elif os.path.exists(f"outputs/04_regularization_ridge_lasso.gif"):
    display(Image(filename=f"outputs/04_regularization_ridge_lasso.gif"))


## Practical tips

- **Always standardize features** first, otherwise the penalty is unfair across units.
- Choose λ by cross-validation (the U-shaped test-error curve on the right).
- Lasso arbitrarily picks one of several correlated features; Elastic Net keeps groups together.


## 🧠 Check Your Understanding

<details>
<summary><b>1. Why does the L1 norm produce exact zeros while L2 does not?</b></summary>
The L1 constraint surface has sharp corners on the coordinate axes (a diamond shape in 2D). When the elliptical OLS loss contours expand, they typically contact the diamond at a vertex where one or more coordinates equal exactly zero.
</details>

<details>
<summary><b>2. Why is feature scaling mandatory before Ridge or Lasso?</b></summary>
Because the penalty $\sum w_j^2$ or $\sum |w_j|$ treats all weights equally. A feature measured in millimeters will have a much smaller coefficient than one in kilometers, causing unscaled penalties to unfairly penalize features simply based on their units.
</details>


## 🧪 Guided Experiments

- Change `alpha` for the elastic net and see how the path shape morphs between ridge and lasso.
- Make training smaller (50 rows) and see how much more regularization helps.
